# Trails temporal LCI case runner

This notebook runs the temporal LCA comparison workflow for the foreground LCI
case studies stored in the OneDrive `trails/data` folder.

The workflow compares two model variants:

- foreground + background temporal distributions
- foreground temporal distributions only

It is parameterized so you can choose which case studies, indicators, depth,
reference year, and output folder to use before running an expensive analysis.

## Modeling convention

The four foreground inventories are interpreted as **full lifetime systems**.
The importer therefore sets each foreground production exchange to the lifetime
output declared in the workbook, for example:

- BEV: `lifetime [km]`
- Polyol: `production volume over lifetime [kg]`
- DACCS: `CO2 capture amount [kg]`
- Marine freight: `total lifetime work [tkm]`

Technosphere exchanges are **not divided** by these lifetime totals. The run
configuration also scales the functional-unit demand to the same lifetime
quantity, and temporal routing accounts for provider production amounts.

Keep these three pieces together when running lifetime systems:

- `scale_demand_amount_by_activity=True`
- `scale_routing_min_amount_by_activity=True`
- the production-aware routing code in `trails.trails`

In [1]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
import csv
import importlib
import sys

from IPython.display import Image, display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name != "trails":
    # If this notebook is opened from dev/, move imports back to the repo root.
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

runner = importlib.import_module("dev.plot_terminal_lci_td_comparison")

/opt/homebrew/Caskroom/miniforge/base/envs/trails/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Edit this cell before running the analysis.

Typical configurations:

- quick smoke test: `CASE_KEYS = ["bev"]`, `METHOD_MODE = "custom"`, one method
- all headline indicators: `CASE_KEYS = ["bev", "polyol", "daccs", "marine"]`,
  `METHOD_MODE = "ef31_headline"`
- one custom EF/IPCC method: `METHOD_MODE = "custom"` and edit `CUSTOM_METHODS`

Leave `RUN_ANALYSIS = False` while editing configuration. Set it to `True` only
when you want to execute the full Trails run.

In [13]:
RUN_ANALYSIS = True

CASE_KEYS = [
    #"bev",
    "polyol",
    "daccs",
    "marine"
]
# CASE_KEYS = ["bev", "polyol", "daccs", "marine"]

METHOD_MODE = "custom"  # "custom", "ef31_headline", or "ipcc2021_incl_biogenic"
CUSTOM_METHODS = [
    "IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)",
    #'EF v3.1 - acidification - accumulated exceedance (AE)',
    #'EF v3.1 - climate change - global warming potential (GWP100)',
    #'EF v3.1 - climate change: biogenic - global warming potential (GWP100)',
    #'EF v3.1 - climate change: fossil - global warming potential (GWP100)',
    #'EF v3.1 - climate change: land use and land use change - global warming potential (GWP100)',
    #'EF v3.1 - ecotoxicity: freshwater - comparative toxic unit for ecosystems (CTUe)',
    #'EF v3.1 - ecotoxicity: freshwater, inorganics - comparative toxic unit for ecosystems (CTUe)',
    #'EF v3.1 - ecotoxicity: freshwater, organics - comparative toxic unit for ecosystems (CTUe)',
    #'EF v3.1 - energy resources: non-renewable - abiotic depletion potential (ADP): fossil fuels',
    #'EF v3.1 - eutrophication: freshwater - fraction of nutrients reaching freshwater end compartment (P)',
    #'EF v3.1 - eutrophication: marine - fraction of nutrients reaching marine end compartment (N)',
    #'EF v3.1 - eutrophication: terrestrial - accumulated exceedance (AE)',
    'EF v3.1 - human toxicity: carcinogenic - comparative toxic unit for human (CTUh)',
    #'EF v3.1 - human toxicity: carcinogenic, inorganics - comparative toxic unit for human (CTUh)',
    #'EF v3.1 - human toxicity: carcinogenic, organics - comparative toxic unit for human (CTUh)',
    'EF v3.1 - human toxicity: non-carcinogenic - comparative toxic unit for human (CTUh)',
    #'EF v3.1 - human toxicity: non-carcinogenic, inorganics - comparative toxic unit for human (CTUh)',
    #'EF v3.1 - human toxicity: non-carcinogenic, organics - comparative toxic unit for human (CTUh)',
    #'EF v3.1 - ionising radiation: human health - human exposure efficiency relative to u235',
    'EF v3.1 - land use - soil quality index',
    'EF v3.1 - material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)',
    'EF v3.1 - ozone depletion - ozone depletion potential (ODP)',
    'EF v3.1 - particulate matter formation - impact on human health',
    'EF v3.1 - photochemical oxidant formation: human health - tropospheric ozone concentration increase',
    'EF v3.1 - water use - user deprivation potential (deprivation-weighted water consumption)',
]

REFERENCE_YEAR = 2025
DEPTH = 5
PLOT_WINDOW_YEARS = 50

OUTPUT_DIR = (
    REPO_ROOT
    / "dev"
    / "notebook_runs"
    / "temporal_lci_case_runner"
)
RESULTS_CSV = OUTPUT_DIR / "scores.csv"

DATAPACKAGE = REPO_ROOT / "dev" / "trails_2026-05-18.zip"
LCIA_JSON = Path("/Users/romain/GitHub/pathways/pathways/data/lcia_ei312.json")

ONEDRIVE_LCI_DIR = Path(
    "/Users/romain/Library/CloudStorage/OneDrive-PaulScherrerInstitut/trails/data"
)
INVENTORY_PATHS = [
    ONEDRIVE_LCI_DIR / "lci-pass_cars.xlsx",
    ONEDRIVE_LCI_DIR / "lci-case-study-ccu_polyol_delayed_release.xlsx",
    ONEDRIVE_LCI_DIR / "lci-case-study-daccs_storage_risk.xlsx",
    ONEDRIVE_LCI_DIR / "lci-case-study-marine_fuel_switch.xlsx",
]

# Full reruns should use resume=False. Use resume=True to skip already complete
# activity/method figures in the selected output directory.
RESUME = False

# Graph HTML files can be large, especially for depth 5 all-TD routing.
WRITE_GRAPH_HTML = True
GRAPH_MIN_EDGE_AMOUNT = 1e-9

# Keep the routing cutoff relative to the represented lifetime system.
ROUTING_MIN_AMOUNT = 1e-12
SCALE_ROUTING_MIN_AMOUNT_BY_ACTIVITY = True
SCALE_DEMAND_AMOUNT_BY_ACTIVITY = True

WIDTH = 1200
HEIGHT = 860
NO_CACHE_INTERPOLATION = True

In [14]:
IPCC_2021_INCL_BIOGENIC = (
    "IPCC 2021 (incl. biogenic CO2) - climate change: total "
    "(incl. biogenic CO2) - global warming potential (GWP100)"
)


def selected_methods() -> tuple[list[str], bool]:
    if METHOD_MODE == "ef31_headline":
        return [], True
    if METHOD_MODE == "ipcc2021_incl_biogenic":
        return [IPCC_2021_INCL_BIOGENIC], False
    if METHOD_MODE == "custom":
        return list(CUSTOM_METHODS), False
    raise ValueError(f"Unknown METHOD_MODE: {METHOD_MODE!r}")


def build_args() -> SimpleNamespace:
    methods, headline_ef = selected_methods()
    return SimpleNamespace(
        datapackage=DATAPACKAGE,
        interpolation_cache_dir=None,
        import_before_interpolation=False,
        inventories=list(INVENTORY_PATHS),
        all_dev_lci_inventories=False,
        all_onedrive_lci_inventories=False,
        case_study_activities=True,
        case_study_activity=list(CASE_KEYS),
        depth=int(DEPTH),
        reference_year=int(REFERENCE_YEAR),
        plot_window_years=int(PLOT_WINDOW_YEARS),
        amount=1.0,
        ei_version="3.12",
        lcia_json=LCIA_JSON,
        methods=methods,
        headline_ef_v31_only=bool(headline_ef),
        max_activities=None,
        output_dir=OUTPUT_DIR,
        results_csv=RESULTS_CSV,
        resume=bool(RESUME),
        width=int(WIDTH),
        height=int(HEIGHT),
        solver_mode="direct",
        fallback_solver_mode="direct",
        iterative_rtol=1e-8,
        iterative_maxiter=None,
        iterative_restart=None,
        routing_min_amount=float(ROUTING_MIN_AMOUNT),
        scale_routing_min_amount_by_activity=bool(
            SCALE_ROUTING_MIN_AMOUNT_BY_ACTIVITY
        ),
        scale_demand_amount_by_activity=bool(SCALE_DEMAND_AMOUNT_BY_ACTIVITY),
        attribute_to_roots=True,
        foreground_attribute_to_roots=False,
        write_graph_html=bool(WRITE_GRAPH_HTML),
        graph_run="all_td",
        graph_min_edge_amount=float(GRAPH_MIN_EDGE_AMOUNT),
        show_progress=False,
        no_cache_interpolation=bool(NO_CACHE_INTERPOLATION),
        interpolation_start_year_offset=-20,
        interpolation_end_year_offset=20,
    )


def validate_config(args: SimpleNamespace) -> None:
    missing = [path for path in [args.datapackage, *args.inventories] if not path.exists()]
    if args.lcia_json is not None and not args.lcia_json.exists():
        missing.append(args.lcia_json)
    if missing:
        raise FileNotFoundError("\n".join(str(path) for path in missing))
    invalid_cases = sorted(set(args.case_study_activity) - set(runner.DEFAULT_CASE_STUDY_ACTIVITY_KEYS))
    if invalid_cases:
        raise ValueError(f"Unknown case key(s): {invalid_cases}")


args = build_args()
validate_config(args)

print("Cases:", ", ".join(args.case_study_activity))
print("Method mode:", METHOD_MODE)
if args.headline_ef_v31_only:
    print("Methods: 16 EF v3.1 headline indicators")
else:
    print("Methods:")
    for method in args.methods:
        print(" -", method)
print("Output directory:", args.output_dir)
print("Results CSV:", args.results_csv)

Cases: bev
Method mode: custom
Methods:
 - IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)
 - EF v3.1 - human toxicity: carcinogenic - comparative toxic unit for human (CTUh)
 - EF v3.1 - human toxicity: non-carcinogenic - comparative toxic unit for human (CTUh)
 - EF v3.1 - land use - soil quality index
 - EF v3.1 - material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)
 - EF v3.1 - ozone depletion - ozone depletion potential (ODP)
 - EF v3.1 - particulate matter formation - impact on human health
 - EF v3.1 - photochemical oxidant formation: human health - tropospheric ozone concentration increase
 - EF v3.1 - water use - user deprivation potential (deprivation-weighted water consumption)
Output directory: /Users/romain/GitHub/trails/dev/notebook_runs/temporal_lci_case_runner
Results CSV: /Users/romain/GitHub/trails/dev/notebook_runs/temporal_lci_case_runner/scores.csv


## Run

This cell performs the expensive calculation. It imports the selected foreground
LCIs twice:

1. all foreground and background temporal distributions
2. foreground temporal distributions only

The run writes one subfolder per activity and one PNG per indicator.

In [15]:
if RUN_ANALYSIS:
    runner.run(args)
else:
    print("Dry run only. Set RUN_ANALYSIS = True in the configuration cell to execute.")

Datapackage: /Users/romain/GitHub/trails/dev/trails_2026-05-18.zip
Excel inventories:
  /Users/romain/Library/CloudStorage/OneDrive-PaulScherrerInstitut/trails/data/lci-pass_cars.xlsx
  /Users/romain/Library/CloudStorage/OneDrive-PaulScherrerInstitut/trails/data/lci-case-study-ccu_polyol_delayed_release.xlsx
  /Users/romain/Library/CloudStorage/OneDrive-PaulScherrerInstitut/trails/data/lci-case-study-daccs_storage_risk.xlsx
  /Users/romain/Library/CloudStorage/OneDrive-PaulScherrerInstitut/trails/data/lci-case-study-marine_fuel_switch.xlsx
EF v3.1 methods: 9
  IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)
  EF v3.1 - human toxicity: carcinogenic - comparative toxic unit for human (CTUh)
  EF v3.1 - human toxicity: non-carcinogenic - comparative toxic unit for human (CTUh)
  EF v3.1 - land use - soil quality index
  EF v3.1 - material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reser

## Inspect Results

Use these cells after a run completes. They do not recalculate anything.

In [ ]:
def load_rows(csv_path: Path = RESULTS_CSV) -> list[dict[str, str]]:
    if not csv_path.exists():
        print(f"No results CSV found: {csv_path}")
        return []
    with csv_path.open(newline="") as handle:
        return list(csv.DictReader(handle))


rows = load_rows()
print("Rows:", len(rows))
print("Activities:", len({row["activity"] for row in rows}))
print("Methods:", len({row["method"] for row in rows}))

for row in rows[:10]:
    print(
        row["activity"],
        "|",
        row["method"],
        "| static:",
        row["static_all_td"],
        "| temporal all TD:",
        row["temporal_cumulative_all_td"],
    )

In [ ]:
def find_figures(
    *,
    activity_contains: str | None = None,
    method_contains: str | None = None,
    output_dir: Path = OUTPUT_DIR,
) -> list[Path]:
    figures = sorted(output_dir.rglob("*.png"))
    if activity_contains:
        figures = [
            path for path in figures
            if activity_contains.lower() in str(path.parent).lower()
        ]
    if method_contains:
        figures = [
            path for path in figures
            if method_contains.lower() in path.name.lower()
        ]
    return figures


figures = find_figures(method_contains="climate_change")
print("Matching figures:", len(figures))
for path in figures[:12]:
    print(path)

if figures:
    display(Image(filename=str(figures[0])))

In [ ]:
def graph_html_paths(output_dir: Path = OUTPUT_DIR) -> list[Path]:
    return sorted(output_dir.rglob("graphs/*.html"))


graphs = graph_html_paths()
print("Graph HTML files:", len(graphs))
for path in graphs:
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"{path} ({size_mb:.1f} MiB)")

## Notes for variations

- To run a single case, set `CASE_KEYS = ["bev"]`, `["polyol"]`,
  `["daccs"]`, or `["marine"]`.
- To run all shortlisted EF v3.1 indicators, set `METHOD_MODE =
  "ef31_headline"`.
- To run one indicator, set `METHOD_MODE = "custom"` and edit
  `CUSTOM_METHODS`.
- To re-use existing figures in the same output folder, set `RESUME = True`.
- To avoid large routing graph HTML files, set `WRITE_GRAPH_HTML = False`.